# Reliability scores

**Input:**  
- Gexf file (from Social Network Visualisation Tool)
- Telegram messages data, raw from data provider (e.g. pandas dataframes or csv/xlsx/json files)

**Dependencies:**  
- Social Network Visualisation Tool
- Disinformation Signals Detection Tool

In [1]:
import os
os.chdir(os.getcwd()) 
cwd = os.getcwd()

import pandas as pd
import numpy as np
from collections import Counter
import networkx as nx

In [4]:
# Read data

df = pd.read_excel(cwd+"\\data\\alt_df.xlsx")
G = nx.read_gexf(cwd+"\\data\\graph.gexf")

Apply WP3 tools to obtain disinformation detection-related labels

List of tools of interest: 
- image/video deepfake detection [output = overall deepfake score] [MEDIA]
- video anomaly detection [output = anomalous score] [MEDIA]
- sensational content detection [output = set of scores (TBD?)] [MEDIA]
- visual text misalignment detection [output = misalignment score] [MEDIA]
- audio deepfake detection [ouptut = deepfake score] [MEDIA]
- audio anomaly detection [output = fakeness score, fakeness decision score (for each segment)] [MEDIA]
- disinformation signals detection [output = list of DisinformationSignal JSON objects] [TEXT]
- verdict generation [output = top_sentences, verdict] [TEXT]

WP4 tools will work only with text-applicable tools, unless the provided data contains media shared in Telegram messages, which has not been the case so far. Therefore, we exclude media-only tools.  
In addition, the output of the verdict generation tool is not appropriate for our pipeline. As seen during the WP4 plenary session in Bucharest, we need synthetical labels for the data, e.g. binary verdicts or disinformation scores. The sentences produced by the verdict generation tool do not seem fit.

## Step 1 - Apply tools to text messages

Let's assume ```disinformation_signals_detection(text)```.

In [ ]:
df['disinfo_signal'] = df['message'].apply(disinformation_signals_detection)

## Step 2 - Aggregate tool outputs at the channel level

In [ ]:
# Initialize dict of channels associated with disinfo score
channel_disinfo_scores = dict.fromkeys(df['to_id'].unique())

# Fill in the dict
# Iterate over channel ids
for channel in channel_disinfo_scores.keys():
    # Retrieve the list of disinfo signals for all messages forwarded to current channel
    disinfo_json_list = df[df['to_id'] == channel]['disinfo_signal']
    # Disinfo score = average number of disinfo signals per message for given channel
    score = np.mean([len(x) for x in disinfo_json_list])
    # Update dict
    channel_disinfo_scores[channel] = score

## Step 3 - Add the scores as a node attribute in the network

In [ ]:
nx.set_node_attributes(G, channel_disinfo_scores, 'disinformation_scores')

## Step 4 - Target areas of high disinfo scores in the network

### (i) for individual channels

In [ ]:
# Evaluate disinfo score threshold empirically, for example:
threshold = np.percentile(channel_disinfo_scores.values, 90)

# Extract channels with disinfo score greater than threshold
disinfo_channels = [k for k,v in channel_disinfo_scores if v >= threshold]

### (ii) for channel neighbors

In [ ]:
# Initialize dict of channels associated with neighbor disinfo score
channel_neighbor_disinfo_scores = dict.fromkeys(df['to_id'].unique())

# Fill in the dict
# Iterate over channel ids
for channel in channel_neighbor_disinfo_scores.keys():
    # Retrieve the disinfo scores for all neighbors
    ngbr_list = nx.all_neighbors(G, channel)
    # Compute the mean disinfo score of channel neighbors
    avg_score = np.mean([v for k,v in channel_disinfo_scores if k in ngbr_list])
    # Update dict
    channel_neighbor_disinfo_scores[channel] = avg_score

In [ ]:
# Evaluate disinfo score threshold empirically, for example:
threshold = np.percentile(channel_neighbor_disinfo_scores.values, 90)

# Extract channels with disinfo score greater than threshold
neighbor_disinfo_channels = [k for k,v in channel_neighbor_disinfo_scores if v >= threshold]

**Output:**  
- ```disinfo_channels``` (list of likely unreliable channel ids)
- ```neighbor_disinfo_channels``` (list of channels whose neighborhood is likely unreliable)